# PyTorch Autograd — Leaf Nodes, Gradients & Vector Backprop

This notebook explores PyTorch's **automatic differentiation engine (autograd)** in depth: leaf vs intermediate nodes, gradient accumulation, backward on vectors, and how to stop gradient tracking.

In [ ]:
import torch

## Autograd — Leaf Nodes

A **leaf tensor** is any tensor you created directly (not the result of an operation on another tracked tensor). In a neural network, **leaf nodes correspond to your learnable weights** — the parameters autograd ultimately computes gradients *for*.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)  # leaf tensor / leaf node

In [ ]:
x.is_leaf

**Leaf nodes = weights.** These are the tensors whose `.grad` gets filled in and used by an optimizer to update the model.

Any tensor produced *from* a leaf tensor via an operation is an **intermediate node** — `is_leaf` is `False` for it, and PyTorch doesn't store `.grad` for it by default (only the computation graph edge).

In [ ]:
y = x + 2
y

In [ ]:
y.is_leaf  # Intermediate node -> False

Calling `.backward()` walks the graph backward from `y` all the way to the leaf `x`, computing `dy/dx` and storing it in `x.grad`. Since `y = x + 2`, `dy/dx = 1`.

In [ ]:
y.backward()  # Backtracking upto leaf node

In [ ]:
x.grad

### Chain rule example: `b = a²`
Here `db/da = 2a`. With `a = 3.0`, we expect `a.grad = 6.0`.

In [ ]:
a = torch.tensor(3., requires_grad=True)
b = a**2
b.backward()
a.grad

### Chained operations: `z = (x + 6)²`
By the chain rule: `dz/dx = 2(x+6)`. With `x = 3.0`, that's `2 * 9 = 18.0`.

In [ ]:
x = torch.tensor(3., requires_grad=True)
y = x + 6
z = y**2
z.backward()
x.grad

### ⚠️ Gradient Accumulation

PyTorch **adds** each new `.backward()` result on top of whatever is already in `.grad` — it does not overwrite by default. Below, `y = x**2` is recomputed and backward'd **three times**, so `x.grad` ends up being `3 × (dy/dx) = 3 × 6 = 18.0`, not `6.0`.

This is exactly why training loops call `optimizer.zero_grad()` before every `.backward()` — otherwise gradients from old batches keep piling up.

In [ ]:
x = torch.tensor(3., requires_grad=True)
y = x**2
y.backward()
y = x**2
y.backward()
y = x**2
y.backward()

x.grad

### Resetting a gradient manually with `.zero_()`
`x.grad.zero_()` clears the accumulated gradient back to `0`, in-place, so the next `.backward()` starts fresh.

In [ ]:
x = torch.tensor(3., requires_grad=True)
y = x**2
y.backward()
x.grad
x.grad.zero_()  # to reset x's gradient back to 0

## Backward on Vectors

`.backward()` normally only works directly on a **scalar** (single number), because it's really computing a gradient of *one output* with respect to inputs. When the output `b` is a **vector**, PyTorch instead needs a *vector-Jacobian product* — you must pass in a tensor the same shape as `b`, telling it how much each element of `b` should contribute.

In [ ]:
a = torch.tensor([1, 2, 3, 5], requires_grad=True, dtype=torch.float32)

In [ ]:
b = a**2
b

In a vector, we **can't** directly call `b.backward()` — instead we pass a matching-shape tensor: `b.backward(torch.tensor([1,1,1,1]))`.

That weighting vector works like a mask/multiplier on each output element:
- `1` → "include this element's gradient contribution"
- `0` → "ignore this element's gradient contribution"

(More precisely: it's the `dL/db` term for some upstream scalar loss `L` — often called the "gradient outputs" or `grad_outputs`. Passing all `1`s is equivalent to treating `L = sum(b)`.)

In [ ]:
b.backward(torch.tensor([1, 1, 1, 1], dtype=torch.float32))
a.grad  # d(a^2)/da = 2a for each element -> [2, 4, 6, 10]

### Selectively including elements
Passing `[0,1,1,0]` zeroes out the gradient contribution from the 1st and 4th elements of `b`, so only the 2nd and 3rd elements of `a.grad` come out non-zero.

In [ ]:
a = torch.tensor([1, 2, 3, 5], requires_grad=True, dtype=torch.float32)

In [ ]:
b = a**2
b

In [ ]:
b.backward(torch.tensor([0, 1, 1, 0], dtype=torch.float32))
a.grad  # only elements 2 and 3 contributed

## Stopping Gradient Tracking

Sometimes you want to compute something **without** it being tracked in the autograd graph — e.g. during inference, or for a constant you don't want gradients flowing through. `with torch.no_grad():` disables tracking for anything computed inside that block.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)


with torch.no_grad():  # z's history will not be stored in the gradient graph
  z = x**3
  print(z.is_leaf)      # True -> untracked tensors are always treated as leaves

a = z + 5               # a's history is also not tracked, since z carries no graph
print(a.is_leaf)

a = x + 5
y = x**5
print(a.is_leaf)         # False -> this a IS tracked (built from tracked x)

y.backward()
x.grad  # d(x^5)/dx = 5x^4 = 5*16 = 80.0

## Multiple Independent Leaf Tensors

When a scalar output depends on **more than one** leaf tensor, calling `.backward()` once fills in `.grad` for *each* of them independently, via partial derivatives.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(5.0, requires_grad=True)

In [ ]:
z = (x**2) + (y**3)
z.backward()
x.grad  # dz/dx = 2x = 4.0

In [ ]:
y.grad  # dz/dy = 3y^2 = 75.0

## `detach_()` — Permanently Cutting a Tensor Out of the Graph

`.detach_()` is an in-place operation that strips a tensor's history from the autograd graph. Anything built from that tensor afterward also won't be tracked, since there's no graph left to trace back through.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(5.0, requires_grad=True)

z = y + 1
z.detach_()  # after this, z's history is removed from the graph;
             # anything built from z afterward is also untracked
print(z)
c = z + 1
c

### Quick recap

| Concept | What it does |
|---|---|
| `requires_grad=True` | marks a leaf tensor for gradient tracking |
| `.is_leaf` | `True` for tensors you created directly (or untracked ones) |
| `.backward()` | computes gradients back to all leaf tensors (scalar output only) |
| `.backward(grad_outputs)` | needed for vector/tensor outputs — a vector-Jacobian product |
| gradient accumulation | `.grad` **adds up** across multiple `.backward()` calls |
| `.grad.zero_()` | manually reset a leaf's accumulated gradient |
| `torch.no_grad()` | context manager to skip tracking entirely (used at inference time) |
| `.detach()` / `.detach_()` | cut a tensor out of the graph (copy vs in-place) |